In [10]:
# Cell 1: Spark session with Kafka package (restart kernel if a session already exists)
from pyspark.sql import SparkSession, functions as F
import time

spark = (SparkSession.builder.appName('OrderDataGenerator')
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print(spark.version)

3.5.9


In [6]:
# Cell 2: Settings
BOOTSTRAP = 'localhost:9092'
TOPIC = 'order-status'
RATE = 5            # new orders per second (all areas)
TICK = 5            # seconds of data generated per loop
RUN_MINUTES = 10    # how long to keep generating

AREAS = ['Udupi', 'Manipal', 'Malpe', 'Kundapura', 'Karkala', 'Brahmavar', 'Kaup', 'Shirva']
HOT_AREAS = ['Manipal', 'Udupi']
AREA_POOL = AREAS + HOT_AREAS + HOT_AREAS      # hot areas get 3x demand
CUISINES = ['North Indian', 'South Indian', 'Chinese', 'Italian', 'Biryani', 'Desserts']

P_CANCEL = 0.05
P_DELAY = 0.15
P_DELAY_HOT = 0.35
P_LATE = 0.03       # event_time is 30-90 sec old when sent
P_DUP = 0.01        # event sent twice
FMT = "yyyy-MM-dd'T'HH:mm:ss"

In [7]:
# Cell 3: Batch generator
def pick(values):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.floor(F.rand() * len(values)) + 1).cast('int'))


def generate_batch(slice_start, n):
    o = (spark.range(n)
        .withColumn('order_id', F.substring(F.expr('uuid()'), 1, 8))
        .withColumn('restaurant_id', F.concat(F.lit('R'), F.lpad((F.floor(F.rand() * 40) + 1).cast('string'), 3, '0')))
        .withColumn('delivery_area', pick(AREA_POOL))
        .withColumn('cuisine', pick(CUISINES))
        .withColumn('placed', F.lit(slice_start) + F.rand() * TICK)
        .withColumn('cancelled', F.rand() < P_CANCEL)
        .withColumn('delay_p', F.when(F.col('delivery_area').isin(HOT_AREAS), P_DELAY_HOT).otherwise(P_DELAY))
        .withColumn('delayed', (~F.col('cancelled')) & (F.rand() < F.col('delay_p')))
        .withColumn('t_prep', F.rand() * 10 + 5)
        .withColumn('t_cancel', F.col('t_prep') + F.rand() * 12 + 3)
        .withColumn('t_delay', F.col('t_prep') + F.rand() * 10 + 5)
        .withColumn('t_out', F.when(F.col('delayed'), F.col('t_delay')).otherwise(F.col('t_prep')) + F.rand() * 10 + 5)
        .withColumn('t_done', F.col('t_out') + F.rand() * 15 + 10))

    def ev(status, off, keep):
        return F.struct(F.lit(status).alias('status'), off.alias('off'), keep.alias('keep'))

    alive = ~F.col('cancelled')
    events = F.array(
        ev('PLACED', F.lit(0.0), F.lit(True)),
        ev('PREPARING', F.col('t_prep'), F.lit(True)),
        ev('CANCELLED', F.col('t_cancel'), F.col('cancelled')),
        ev('DELAYED', F.col('t_delay'), F.col('delayed')),
        ev('OUT_FOR_DELIVERY', F.col('t_out'), alive),
        ev('DELIVERED', F.col('t_done'), alive))

    keep_cols = ['order_id', 'restaurant_id', 'delivery_area', 'cuisine', 'placed']
    e = (o.select(*keep_cols, F.explode(events).alias('e'))
          .select(*keep_cols, 'e.status', 'e.off', 'e.keep')
          .filter(F.col('keep'))
          .withColumn('late', F.when(F.rand() < P_LATE, F.rand() * 60 + 30).otherwise(F.lit(0.0)))
          .withColumn('event_ts', F.col('placed') + F.col('off') - F.col('late'))
          .withColumn('event_time', F.from_unixtime(F.col('event_ts').cast('long'), FMT))
          .withColumn('estimated_delivery_time', F.from_unixtime((F.col('placed') + 1800).cast('long'), FMT))
          .withColumn('dup', F.rand() < P_DUP)
          .select('order_id', 'restaurant_id', 'delivery_area', 'status',
                  'estimated_delivery_time', 'cuisine', 'event_time',
                  F.explode(F.when(F.col('dup'), F.array(F.lit(1), F.lit(2))).otherwise(F.array(F.lit(1)))).alias('copy')))

    return e.select(
        F.col('order_id').alias('key'),
        F.to_json(F.struct('order_id', 'restaurant_id', 'delivery_area', 'status',
                           'estimated_delivery_time', 'cuisine', 'event_time')).alias('value'))

In [8]:
# Cell 4: Generate and send to Kafka in a loop (stop with the kernel Interrupt button)
batches = int(RUN_MINUTES * 60 / TICK)
n = int(RATE * TICK)
print(f'Sending {n} new orders every {TICK}s for {RUN_MINUTES} minutes to topic {TOPIC}')

try:
    for i in range(batches):
        out = generate_batch(time.time(), n)
        (out.write.format('kafka')
            .option('kafka.bootstrap.servers', BOOTSTRAP)
            .option('topic', TOPIC)
            .save())
        time.sleep(TICK)
        if (i + 1) % 6 == 0:
            print(f'batch {i + 1}/{batches} sent')
except KeyboardInterrupt:
    print('Stopped by user')
print('Done')

Sending 25 new orders every 5s for 10 minutes to topic order-status
batch 6/120 sent
batch 12/120 sent
batch 18/120 sent
batch 24/120 sent
batch 30/120 sent
batch 36/120 sent
batch 42/120 sent
batch 48/120 sent
batch 54/120 sent
batch 60/120 sent
batch 66/120 sent
batch 72/120 sent
batch 78/120 sent
batch 84/120 sent
batch 90/120 sent
batch 96/120 sent
batch 102/120 sent
batch 108/120 sent
batch 114/120 sent
batch 120/120 sent
Done


In [9]:
# Cell 5 (optional): Check what is stored in the topic
(spark.read.format('kafka')
    .option('kafka.bootstrap.servers', BOOTSTRAP)
    .option('subscribe', TOPIC)
    .option('startingOffsets', 'earliest')
    .load()
    .selectExpr('CAST(key AS STRING) AS key', 'CAST(value AS STRING) AS value')
    .show(10, False))

+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key     |value                                                                                                                                                                                                      |
+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|75db4ed6|{"order_id":"75db4ed6","restaurant_id":"R011","delivery_area":"Brahmavar","status":"PLACED","estimated_delivery_time":"2026-10-14T10:26:42","cuisine":"South Indian","event_time":"2026-10-14T09:56:42"}   |
|d51d3176|{"order_id":"d51d3176","restaurant_id":"R009","delivery_area":"Malpe","status":"PLACED","estimated_delivery_time":"2026-10-14T10:2